# Tutorial 150 — Define project types and route intents

[Tutorial 140](./140_logging_trace.ipynb) followed one complete Project Chat
operation through its execution trace. This tutorial moves one level higher in
the architecture: you will define what kind of project QAChat is running and how
different user requests should be handled.

A **project type** is a declarative JSON configuration that defines the behavior
of a family of QAChat projects. It specifies:

- which source files the project accepts by default;
- the instructions shared by its answers;
- which **intents** the project recognizes;
- how much conversation history each intent uses;
- how each intent obtains evidence, or whether it invokes a tool.

QAChat includes built-in project types for code and literature. You will inspect
their definitions, follow the routing of a question to an intent, create a small
project type for meeting notes, validate it, and test its intents with the real
router model.

Earlier tutorials primarily used QAChat through the public `qachat` API. From
this point onward, the series also examines QAChat's extension architecture.
Some definitions used to create project types, routing rules, and tools therefore
live in their owning submodules rather than in the top-level API.

[Tutorial 160](./160_create_custom_tool.ipynb) continues this extension layer by
showing how an intent can invoke a Python tool for operations that need more
control than retrieval followed by ordinary answer generation.


## 1. Prepare QAChat

Each tutorial is independently runnable. These cells locate the repository,
make the local package importable, and call `prepare_notebook()` to verify the
project `.venv` environment. If Jupyter asks you to select the printed `.venv`
kernel, select it and rerun the cell.

In [ ]:
from pathlib import Path

root = Path.cwd()
if not (root / "qachat").is_dir():
    root = root.parent

print(f"QAChat root: {root}")

In [ ]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [ ]:
from setup.notebook import prepare_notebook

root = prepare_notebook(root)

In [ ]:
import qachat

## 2. Inspect the built-in project types

Project-type definitions live as JSON files in `qachat/project_types/`.

The `load_project_types()` function reads the definitions in that directory and
validates them before they are used. The file name becomes the project-type ID,
so `code.json` defines `code` and `literature.json` defines `literature`.

Start by listing the available definitions and the main properties that
distinguish them.


In [ ]:
from qachat.project_type import PROJECT_TYPES_DIR, load_project_types

print(f"Folder: {PROJECT_TYPES_DIR}\n")

for kind, item in load_project_types().items():
    print(f"{kind:12} {item.display_name}")
    print(f"{'':12} reads {', '.join(item.extensions)}")
    print(f"{'':12} {len(item.intents)} intents\n")

## 3. Inspect the anatomy of a project type

Open
[`qachat/project_types/literature.json`](../qachat/project_types/literature.json)
in the QAChat repository. The configuration separates project-wide behavior
from the individual intents that handle different kinds of requests.
Its top-level fields are:

| Field | What it controls |
|---|---|
| `display_name` | The human-readable name of the project type. |
| `description` | A short description of the project sources that is also shown to the router. |
| `extensions` | The file extensions that a new project of this type reads by default. |
| `max_file_bytes` | The default maximum size of an individual source file. |
| `retrieval_instruction` | The task description added to embedding queries during retrieval. |
| `system_prompt` | The base instructions used for answers in this project type; intent-specific instructions can be added to them. |
| `answer_guidance` | Default answer-length and formatting guidance. Intent-specific `answer_guidance` can override this for individual request types. |
| `capabilities.code_graph` | Whether QAChat builds and searches the code graph: `true`, `false`, or `"auto"`. |
| `history.recent_turns` | The default number of earlier conversation turns retained as context. |
| `default_intent` | The intent used when routing is disabled or does not produce a usable intent. |
| `intents` | The request types supported by the project type. Each intent can define its own instructions, history policy, retrieval behavior, tool, and routing shortcuts. |
| `welcome_prompts` | Example welcome prompts for this project type. These can serve as seeds for generating project-specific welcome prompts from the project's actual sources. |
| `suggestions` | Example requests for this project type. These can serve as seeds for generating project-specific suggestions from the project's actual sources. |

Read the raw JSON first. Later cells load the same information through QAChat's
validated project-type objects.


In [ ]:
import json

project_type_literature = json.loads(
    (PROJECT_TYPES_DIR / "literature.json").read_text(encoding="utf-8")
)

Inspect its top-level fields before looking at the intents.

In [ ]:
for key, value in project_type_literature.items():
    if key == "intents":
        print(f"{key:25}: There are {len(value)} intents:")
        for i, intent in enumerate(value, start=1):
            print(f"{'':25}  {i:2}. {intent}")
    else:
        print(f"{key:25}: {str(value)[:70]}")

## 4. Inspect the intents

An **intent** describes one kind of request that can occur within a project
type. Each entry in `intents` has an **intent ID**, such as
`question_answering` or `fact_finding`, and a configuration that tells QAChat
how to recognize and handle that request.

The router compares the user's question with the intent descriptions and
selects one intent for the turn. The `description` is therefore part of the
routing contract: it should state what belongs to the intent, include
representative requests where useful, and distinguish the intent from nearby
alternatives.

Once an intent has been selected, its other fields determine how QAChat handles
the request. Not every intent needs every field.

| Field | What it controls |
|---|---|
| `description` | Describes the requests that belong to the intent. The router uses this field when selecting an intent. |
| `system_prompt` | Adds intent-specific instructions to the project type's system prompt. A tool-based intent can omit it because the tool produces the result. |
| `answer_guidance` | Optionally specifies answer-length or formatting guidance for this intent. When present, it takes precedence over more general answer guidance for the turn. |
| `history.policy` | Controls which earlier conversation turns are available: `none`, `recent`, or `relevant`. `relevant` includes recent history together with turns identified as related to the current question. |
| `retrieval.mode` | Selects how evidence is obtained: `search` performs retrieval, `document` reads one whole source within a token budget, `reuse` reuses evidence from a related earlier turn, and `none` performs no retrieval. |
| `retrieval.strategy` | Selects the retrieval strategy when `mode` is `search`. Currently QAChat supports `hybrid`. |
| `retrieval.include_related_evidence` | When searching, also includes evidence cited by related earlier turns so that follow-up questions can retain their earlier sources. |
| `retrieval.max_tokens` | Sets the evidence budget for `document` retrieval. Longer documents are reduced to the parts most relevant to the question. |
| `tool` | Selects a registered Python tool and provides its settings. A tool replaces the normal retrieval-and-answering path, so tool intents use `retrieval.mode: "none"`. |
| `shortcuts` | Optional regular-expression patterns for requests that can be classified deterministically. A matching message selects the intent without calling the router model. |

Fields that are omitted receive sensible defaults. For example, an ordinary
intent defaults to `relevant` history and `search` retrieval using the
`hybrid` strategy, whereas an intent with a `tool` defaults to no retrieval.

Inspect the built-in literature intents. Notice how their descriptions define
the routing boundaries, while the remaining fields determine what QAChat does
after each intent has been selected.


In [ ]:
from qachat.project_type import project_type

project_type_literature = project_type("literature")

print(f"{'Intent':24} {'History':10} Action")
print("-" * 65)

for intent_id, intent in project_type_literature.intents.items():
    if intent.tool:
        action = f"tool: {intent.tool.name}"
    else:
        action = f"retrieval: {intent.retrieval_mode}"

    print(f"{intent_id:24} " f"{intent.history_policy:10} " f"{action}")

## 5. See how QAChat selects an intent

The intents define the possible ways QAChat can handle a request. Before any of
their retrieval, history, or tool settings can be used, QAChat must decide
which intent applies to the current question.

This process is called **routing**.

QAChat first checks the intent `shortcuts`. These are deterministic patterns
for simple requests that do not require a model decision. If no shortcut
matches, QAChat asks the router model to classify the question.

The router receives:

- the project type and its description;
- the IDs and descriptions of the available intents;
- compact representations of earlier turns that are eligible for routing; and
- the new question.

It returns a small structured decision containing:

- `intent`: the selected intent;
- `standalone`: whether the question can be understood without earlier turns;
- `related_turns`: when needed, the earlier turns on which the question
  depends.

QAChat validates this decision before using it. For example, an unknown intent
falls back to the project's default intent, and invalid references to earlier
turns are removed.


You can inspect this process without loading a real language model. The small
stand-in model below prints the routing prompt it receives and returns a
response that we choose in advance.


In [ ]:
class PrintingRouter:
    """Minimal stand-in for the model used by the QAChat router."""

    def __init__(self, response):
        # Predefined response returned instead of calling a real model.
        self.response = response

        # Attributes expected by the QAChat model interface.
        self.messages = []
        self.thinking = False
        self.temperature = 0.0

    def restore_history(self, messages):
        """Store the conversation history supplied by the router."""
        self.messages = list(messages)

    def configure_generation(self, settings):
        """Accept generation settings without changing the stand-in model."""
        pass

    def reply(self, message, *, max_new_tokens=None):
        """Print the routing prompt and return the predefined response."""
        print(message)
        return self.response

Consider a request to read a particular paper in full:

> *Read the Smith paper in full and tell me its limitations.*

From the intents inspected above, we expect this request to select
`document_reading`. The question also identifies what it is about, so it does
not depend on an earlier conversation turn: it is `standalone`.

Configure the stand-in router to return that decision.


In [ ]:
router = PrintingRouter('{"intent": "document_reading", "standalone": true}')

Route the question. The printed prompt shows the information QAChat gives
to the router model.


In [ ]:
from qachat.answering import route_question

decision = route_question(
    router,
    project=project_type_literature,
    turns=[],
    question="Read the Smith paper in full and tell me its limitations",
)

Inspect the validated routing decision.


In [ ]:
print(f"Intent:     {decision.intent}")
print(f"Standalone: {decision.standalone}")
print(f"Related:    {decision.related_turn_ids}")

The router selected `document_reading`. QAChat can now use the
configuration of that intent to determine how the request should be handled—in
this case, by reading a document rather than performing ordinary search
retrieval.

`standalone` is `True` because the question contains enough information to be
understood on its own. For a follow-up such as *What about its limitations?*,
the router could instead mark the question as non-standalone and identify the
earlier turn it refers to.

### Shortcuts bypass the router model

Not every request requires model-based classification. An intent can define
`shortcuts` for requests that can be recognized reliably from their text.

The literature project defines a shortcut for simple greetings. Because
shortcuts are checked before model-based routing, a matching greeting selects
its intent immediately. The stand-in router therefore prints nothing in the
next cell.


In [ ]:
decision = route_question(
    router,
    project_type_literature,
    [],
    "Hello there!",
)

print(f"Intent:   {decision.intent}")
print(f"Shortcut: {decision.shortcut}")

At this point, the complete relationship is visible:

**project type → intents → routing → selected intent → configured behavior**

![Project type and intent routing](./150_project_types_flow.svg)

You can now define a new project type and see how these pieces are specified
together.


## 6. Create a new project type

Now define a project type for a folder with Markdown meeting notes. The project
should answer ordinary questions, list decisions and action items, read a whole
meeting when requested, follow up using earlier evidence, and respond to simple
greetings without retrieval.

This example is different from the built-in code and literature projects, but
it uses the same declarative schema. No new Python behavior is needed.

The five intents will exercise different combinations of history and evidence:

- `question_answering` searches the notes and can use relevant conversation;
- `decision_log` searches specifically for decisions and action items;
- `meeting_reading` reads a complete requested document rather than ranked
  passages;
- `conversation_followup` reuses evidence from earlier turns;
- `small_talk` needs neither history nor evidence and uses a shortcut.

Build the configuration incrementally so that the project-wide settings and
intent-specific behavior remain visible.

### 6.1 Define the project-wide settings

Start with the settings that apply to the project type as a whole: its name and
description, accepted files, retrieval instructions, answer instructions,
capabilities, conversation history, and example suggestions.

Begin with the name shown in the app and a description of the sources.


Start with the name shown in the app and a description of the sources.

In [ ]:
meeting_notes = {
    "display_name": "Meeting notes",
    "description": "Meeting notes written as Markdown or text files.",
}

Choose the file types the project accepts and the maximum size of each
source file.


In [ ]:
meeting_notes.update(
    {"extensions": [".md", ".txt"], "max_file_bytes": 2_000_000}
)

Tell the embedding model what a search query is trying to find.


In [ ]:
meeting_notes.update(
    {
        "retrieval_instruction": (
            "Given a question about meetings, "
            "retrieve the notes that answer it"
        )
    }
)

Set the instructions shared by every ordinary answer in this project type.
Individual intents can add more specific instructions.


In [ ]:
meeting_notes.update(
    {
        "system_prompt": (
            "You answer questions about the team's meeting notes. "
            "Use only the notes you are given and cite them like [1]. "
            "Name the meeting date when it matters."
        )
    }
)

Declare project-wide capabilities. Meeting notes do not need a code graph.


In [ ]:
meeting_notes.update({"capabilities": {"code_graph": False}})

Choose how many recent turns are retained as project-wide conversation
context.


In [ ]:
meeting_notes.update({"history": {"recent_turns": 2}})

Finally, add example requests appropriate for this type of project.


In [ ]:
meeting_notes.update(
    {
        "suggestions": [
            "What did we decide about …?",
            "Who agreed to do what last week?",
            "Read the meeting of … in full",
        ]
    }
)

### 6.2 Define the intents

The project-wide settings establish the common behavior. Now define the
request types that can occur within a meeting-notes project.

Create an empty intent collection, then add the five intents one at a time.


In [ ]:
meeting_notes["intents"] = {}

Ordinary questions search the notes. Relevant conversation history helps
QAChat interpret follow-up context when necessary.


In [ ]:
meeting_notes["intents"]["question_answering"] = {
    "description": (
        "A question answered from the notes "
        "('when is the launch?', 'what did Ana say about the budget?')."
    ),
    "system_prompt": "Answer from the cited notes.",
    "history": {"policy": "relevant"},
    "retrieval": {"mode": "search", "strategy": "hybrid"},
}

A decision-log request also searches the notes, but it asks specifically
for decisions and action items and does not use earlier turns.


In [ ]:
meeting_notes["intents"]["decision_log"] = {
    "description": (
        "A request for decisions or action items: what was decided, agreed, "
        "or assigned, and to whom ('what did we decide about hiring?', "
        "'list my action items')."
    ),
    "system_prompt": (
        "List each decision or action item on its own line "
        "with the meeting date and the people involved."
    ),
    "history": {"policy": "none"},
    "retrieval": {"mode": "search"},
}

A whole-meeting request uses document reading rather than ranked passages.
Large documents can be budget-limited in this mode;
[Tutorial 160](./160_create_custom_tool.ipynb) introduces a separate
batched-review tool.


In [ ]:
meeting_notes["intents"]["meeting_reading"] = {
    "description": (
        "A request about one whole meeting, read in full "
        "('go through the 12 May meeting', "
        "'summarize the whole kickoff notes'). "
        "Not for a question about one detail."
    ),
    "system_prompt": "Cover the whole meeting in order.",
    "history": {"policy": "none"},
    "retrieval": {"mode": "document"},
}

A conversational follow-up reuses evidence found for earlier answers.


In [ ]:
meeting_notes["intents"]["conversation_followup"] = {
    "description": (
        "A follow-up about an earlier answer that its evidence can answer "
        "('why?', 'who said that?')."
    ),
    "system_prompt": "Answer from the earlier evidence.",
    "history": {"policy": "relevant"},
    "retrieval": {"mode": "reuse"},
}

Small talk needs no evidence. Shortcut patterns skip the router when the
whole message matches.


In [ ]:
meeting_notes["intents"]["small_talk"] = {
    "description": "Greetings and thanks, not about the notes.",
    "system_prompt": "Reply briefly and offer help.",
    "history": {"policy": "none"},
    "retrieval": {"mode": "none"},
    "shortcuts": ["(hi|hello|hey|thanks|thank you)( qachat)?"],
}

Inspect the five intent IDs before choosing the fallback behavior.


In [ ]:
list(meeting_notes["intents"])

Choose the fallback intent used when routing does not produce a usable
intent.


In [ ]:
meeting_notes["default_intent"] = "question_answering"

### 6.3 Save and load the project type

The configuration is now complete. Save it as `meeting_notes.json` in a
temporary folder rather than modifying QAChat's installed project types. The
file name becomes the project-type ID.

Create the folder first.


In [ ]:
folder = root / "sandbox" / "tutorial_150_project_types"
folder.mkdir(parents=True, exist_ok=True)

Write the configuration as formatted JSON.


In [ ]:
path = folder / "meeting_notes.json"
path.write_text(json.dumps(meeting_notes, indent=2), encoding="utf-8");

Open the generated [`meeting_notes.json`][meeting-notes-json] to inspect the
complete configuration assembled above. The link becomes available after the
previous cell has written the file.

[meeting-notes-json]: ../sandbox/tutorial_150_project_types/meeting_notes.json

Now load the saved type. Loading validates its project-wide fields and intents.


In [ ]:
custom = load_project_types(folder)["meeting_notes"]

print(custom.display_name)

### 6.4 Check validation errors

Project types are validated when they are loaded. Errors identify the project
type, intent, and invalid field so that configuration mistakes can be found
quickly.

Create a separate invalid copy in which one intent requests a retrieval mode
that does not exist.


In [ ]:
broken = json.loads(json.dumps(meeting_notes))
broken["intents"]["decision_log"]["retrieval"]["mode"] = "everything"

Keep the invalid example in its own folder so that the valid configuration
remains available.


In [ ]:
broken_folder = folder / "broken"
broken_folder.mkdir(exist_ok=True)
(broken_folder / "meeting_notes.json").write_text(json.dumps(broken));

Load the invalid definition to see the validation error.


In [ ]:
try:
    load_project_types(broken_folder)
except ValueError as error:
    print(error)

## 7. Test the project type with the real router

The deterministic model showed the routing contract. Now load the normal chat
model and classify several questions against the new project type.

This is an important validation step when designing intents: the router sees the
intent descriptions, so representative questions should separate cleanly. If a
request is routed incorrectly, improve the descriptions before changing the
rest of the project behavior.

Resolve the requested backend first, choose the corresponding default chat
model, and then load it. The first run may download the model if it is not
already cached.


In [ ]:
backend = qachat.choose_backend(qachat.BACKEND_AUTO)
print("Backend:", backend)

Choose the default chat model for that backend.

In [ ]:
model_id = (
    qachat.DEFAULT_CHAT_MODEL_MLX
    if backend == qachat.BACKEND_MLX
    else qachat.DEFAULT_CHAT_MODEL_TORCH
)
print("Router model:", model_id)

Load the model; the first run may download it.

In [ ]:
router_model = qachat.ChatSession(
    model_id,
    backend=backend,
    thinking=False,
    temperature=0.0,
    allow_download=True,
)

In [ ]:
question = "When is the product launch?"
decision = route_question(router_model, custom, [], question)
print(decision.intent)

Compare a few requests to check whether the descriptions separate the intents.

In [ ]:
questions = [
    "What did we decide about hiring a designer?",
    "Go through the whole 12 May meeting",
    "thanks",
]
for question in questions:
    decision = route_question(router_model, custom, [], question)
    print(f"{decision.intent:20} {question}")

Release the model when you have finished testing.

In [ ]:
router_model.close()

If a question lands in the wrong intent, improve the descriptions rather than
adding special-case routing logic. Add a representative example to the intended
intent or clarify an exclusion in a competing intent.

## 8. Add the project type to QAChat

The example above was loaded from a temporary folder so that the installed
project types remained unchanged. To make a finished project type available to
QAChat, place its JSON file in `qachat/project_types/` and restart the
application.

The file name becomes the project-type ID, its `display_name` appears in the
project-type selector, and its `suggestions` appear in an empty chat. The
built-in `code.json` and `literature.json` definitions should remain in that
directory.

At this point, the behavior of a new family of QAChat projects can be extended
through configuration alone: project-wide instructions define the domain,
intents classify requests, and each intent selects its history and evidence
strategy. The remaining extension point is for operations that need explicit
Python behavior rather than ordinary retrieval and answer generation.


In [ ]:
installed = PROJECT_TYPES_DIR / "meeting_notes.json"
print(f"To install, copy\n  {path}\nto\n  {installed}")

## Continue the tutorial series

**Previous:** [Tutorial 140 — Inspect a QAChat execution trace](./140_logging_trace.ipynb)

**Next:** [Tutorial 160 — Create a custom QAChat tool](./160_create_custom_tool.ipynb)

Tutorial 160 shows how an intent can invoke a registered tool, using QAChat's
research-briefing workflow to connect project-type configuration with explicit,
testable Python operations.
